# Aminos sur GPU gratuit (Google Colab)

Traduit un document avec un **modèle local sur le GPU gratuit de Colab** (en général un NVIDIA T4 ou P100, environ 15 Go de mémoire).
Le modèle tourne **dans** la machine Colab via Ollama ; rien n'est exposé à l'extérieur.

## À lire avant de commencer
- **Confidentialité : Colab est le cloud de Google.** Le document et sa traduction sont traités sur des machines de Google. N'utilisez pas ce carnet pour des pièces confidentielles ou couvertes par le secret professionnel.
- **Pas de garantie** : la disponibilité du GPU gratuit et la durée de session varient selon la demande ; la session peut s'arrêter à tout moment. Téléchargez vos résultats au fur et à mesure.
- La mémoire de traduction est perdue à la fin de la session, sauf si vous cochez l'option Google Drive plus bas.
- Qualité : les modèles de cette taille ne valent pas un grand modèle pour l'arabe juridique. Relisez toujours le résultat.

**Étapes** : *Exécution > Modifier le type d'exécution > GPU T4*, puis exécutez les cellules dans l'ordre.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
REF = "main"  #@param {type:"string"}
# "main" once the work is merged; before that, use the branch name (e.g. claude/stoic-rubin-iq1po6)
import subprocess, sys
spec = f"aminos[docx,pdf,ocr] @ git+https://github.com/aminecopilote/Aminos@{REF}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
print("Aminos installé")

In [ ]:
import json, subprocess
r = subprocess.run(["aminos", "colab-plan"], capture_output=True, text=True)
plan = json.loads(r.stdout)
print(json.dumps(plan, indent=2, ensure_ascii=False))
if plan["gpu"] == "none":
    raise SystemExit("Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4, puis relancez.")
MODEL = plan["model"]

In [ ]:
import os, shutil, subprocess, time, urllib.request
if not shutil.which("ollama"):
    # Installateur officiel d'ollama.com, exécuté dans cette machine virtuelle jetable
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
log = open("/content/ollama.log", "w")
server = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)   # écoute 127.0.0.1 seulement
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama n'a pas démarré, voir /content/ollama.log")
subprocess.run(["ollama", "pull", MODEL], check=True)
print("Modèle prêt :", MODEL)

In [ ]:
USE_DRIVE = False  #@param {type:"boolean"}
TM_PATH = "/content/tm.sqlite"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/aminos", exist_ok=True)
    TM_PATH = "/content/drive/MyDrive/aminos/tm.sqlite"   # mémoire de traduction conservée dans votre Drive
print("Mémoire de traduction :", TM_PATH)

In [ ]:
SRC = "arabe"      #@param ["arabe", "français", "anglais", "néerlandais"]
TGT = "français"   #@param ["français", "arabe", "anglais", "néerlandais"]
MODE = "normal"    #@param ["fast", "normal", "hard"]
from google.colab import files
print("Déposez le document à traduire (.docx, .pdf ou .txt)")
up = files.upload()
doc = next(iter(up))
print("Reçu :", doc)

In [ ]:
GLOSSAIRE = ""
import pathlib
if input("Importer un glossaire ? (o/N) ").strip().lower() == "o":
    g = files.upload()
    if g:
        GLOSSAIRE = "/content/glossaire.json"
        subprocess.run(["aminos", "glossary-import", f"/content/{next(iter(g))}", "-o", GLOSSAIRE], check=True)

In [ ]:
import pathlib
out = f"/content/{pathlib.Path(doc).stem}_{TGT}.docx"
cmd = ["aminos", "translate", f"/content/{doc}", "-s", SRC, "-t", TGT, "-p", "ollama", "--model", MODEL,
       "-m", MODE, "--tm", TM_PATH, "-o", out]
if GLOSSAIRE:
    cmd += ["-g", GLOSSAIRE]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:])
r.check_returncode()

In [ ]:
files.download(out)

## Quand vous avez fini
*Exécution > Se déconnecter et supprimer l'environnement d'exécution* : le document, la traduction et le modèle sont effacés de la machine Colab.
Si vous avez utilisé Google Drive, la mémoire de traduction reste dans `MyDrive/aminos/`.